In [5]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

## Load database

In [6]:
# Setup pandas display options
pd.set_option('display.max_columns', None)

BASE_PATH = Path.cwd().parent
DATA_PATH = BASE_PATH / "researchment" / "database" / "indonesia_fintech_paylater.parquet"

# Load the dataset
df = pd.read_parquet(DATA_PATH)
df.head()

,borrower_id,age,gender,province,education_level,employment_status,monthly_income_idr,regional_minimum_wage_umr_idr,num_active_lending_apps,total_loan_outstanding_idr,credit_limit_granted_idr,loan_tenor_months,daily_interest_rate_pct,primary_loan_purpose,debt_to_income_ratio,income_to_umr_ratio,credit_limit_utilization_ratio,monthly_installment_estimate_idr,debt_service_ratio,disposable_income_after_debt_idr,days_past_due,financial_distress_index,twp90_default_status,primary_default_root_cause,cot_financial_psychology_rationale,confidence_score,risk_score,risk_short,is_high_risk,distress_bin
0,ID_DEB_00264B68D1,24,Female,Bali,High School / SMK,Micro-Entrepreneur / UMKM,6420000,2950000,2,8380000,12280000,1,0.35,Daily Living & Emergency Buffer,1.31,2.18,0.6824,9259900.0,1.4424,-2839900.0,67,3.97,Doubtful-DPD61-90,High-Interest-Rate-Compounding,"Debitur 'ID_DEB_00264B68D1' (Female, usia 24 t...",0.961,3,Doubt(61-90),True,"(3.686, 4.588]"
1,ID_DEB_0DE3583B12,28,Female,Bali,Bachelor (S1),Formal Private Employee,8630000,2950000,1,20110000,29060000,6,0.30,Daily Living & Emergency Buffer,2.33,2.93,0.6920,5161567.0,0.5981,3468433.0,11,4.87,Special-Mention-DPD1-30,High-Interest-Rate-Compounding,"Debitur 'ID_DEB_0DE3583B12' (Female, usia 28 t...",0.975,1,SM(1-30),False,"(4.588, 5.49]"
2,ID_DEB_0F661FADC0,20,Female,Bali,High School / SMK,Civil Servant / BUMN,5720000,2950000,4,14960000,23900000,1,0.32,Daily Living & Emergency Buffer,2.62,1.94,0.6259,16396160.0,2.8665,-10676160.0,180,6.07,Default-Macet-TWP90,High-Interest-Rate-Compounding,"Debitur 'ID_DEB_0F661FADC0' (Female, usia 20 t...",0.990,4,Default Rate(>90),True,"(5.49, 6.392]"
3,ID_DEB_10C74D3F21,31,Female,Bali,Bachelor (S1),Civil Servant / BUMN,11780000,2950000,3,7620000,26130000,1,0.26,Daily Living & Emergency Buffer,0.65,3.99,0.2916,8214360.0,0.6973,3565640.0,24,2.35,Special-Mention-DPD1-30,Compulsive-FOMO-Spending,"Debitur 'ID_DEB_10C74D3F21' (Female, usia 31 t...",0.977,1,SM(1-30),False,"(1.882, 2.784]"
4,ID_DEB_12CFCAE01F,22,Male,Bali,Bachelor (S1),Formal Private Employee,6490000,2950000,2,11080000,16420000,9,0.23,Daily Living & Emergency Buffer,1.71,2.20,0.6748,1995631.0,0.3075,4494369.0,0,4.43,Current-Performing,Compulsive-FOMO-Spending,"Debitur 'ID_DEB_12CFCAE01F' (Male, usia 22 tah...",0.980,0,Current,False,"(3.686, 4.588]"


### - **🎯 Task 2: Gen Z Debt-to-Income (DTI) & Financial Distress Index Forecasting**
###    - **Objective:** Forecast continuous `debt_to_income_ratio` and `financial_distress_index` from demographic, educational, and app multiplicity features to establish pre-emptive credit limits and debt caps.
###    - **Recommended ML Models:** Bayesian Ridge Regression, CatBoost Regressor, Deep learning tensorflow.keras.
###    - **Evaluation Metrics:** `Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), R-Squared (R²)`

## Feature Engineering

In [7]:
# Add datetime features on df
np.random.seed(42)

# Define a realistic time window (e.g. past 2 years up to today)
start_date = pd.to_datetime('2024-11-05')
end_date = pd.to_datetime('2026-11-05')

# Generate 10000 random timestamps between these dates
random_timestamps = np.random.randint(int(start_date.timestamp()), int(end_date.timestamp()), size=len(df))

# Add timestamp features to the DataFrame
df['date'] = pd.to_datetime(random_timestamps, unit='s')
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['hour'] = df['date'].dt.strftime('%H:%M:%S')

# Move the date column to the first position
date_col = "date"
columns_move = df.columns.tolist()
columns_move.remove(date_col)

borrower_index = columns_move.index("borrower_id")
columns_move.insert(borrower_index + 1, date_col)

df = df[columns_move]

# Sort the DataFrame by the new 'date' column
df = df.sort_values(by='date').reset_index(drop=True)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 35 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   borrower_id                         10000 non-null  object        
 1   date                                10000 non-null  datetime64[ns]
 2   age                                 10000 non-null  int64         
 3   gender                              10000 non-null  object        
 4   province                            10000 non-null  object        
 5   education_level                     10000 non-null  object        
 6   employment_status                   10000 non-null  object        
 7   monthly_income_idr                  10000 non-null  int64         
 8   regional_minimum_wage_umr_idr       10000 non-null  int64         
 9   num_active_lending_apps             10000 non-null  int64         
 10  total_loan_outstanding_